# Week 7: Task 4 - Iris Flower Classification
## Data Preprocessing & Partitioning (Wednesday Deliverable)

**Internship:** Arch Technologies, Machine Learning Domain, Month 2  
**Task:** Task 4 — Iris Flower Classification (First Half)  
**Author:** Sharjeel Shahzad  

### Overview & Objectives
Following exploration and visualization, the Wednesday deliverable addresses **Data Preprocessing**:
1. Encoding the categorical target `Species` into numerical labels using `sklearn.preprocessing.LabelEncoder`.
2. Inspecting and handling potential outliers via the Interquartile Range (IQR) method across all four morphological features.
3. Performing an 80/20 train/test split with `random_state=42`.
4. Exporting the preprocessed datasets to:
   - `outputs/X_train.csv`
   - `outputs/X_test.csv`
   - `outputs/y_train.csv`
   - `outputs/y_test.csv`
5. Verifying data integrity for the downstream baseline model (Thursday).


## 1. Environment Setup & Data Loading


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Resolve paths
data_path = Path("../data/iris-dataset.csv")
if not data_path.exists():
    data_path = Path("week-7/data/iris-dataset.csv")

outputs_dir = Path("../outputs")
if not outputs_dir.exists():
    outputs_dir = Path("week-7/outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)

# Load data and drop Id
df = pd.read_csv(data_path)
if "Id" in df.columns:
    df.drop(columns=["Id"], inplace=True)

print(f"Loaded raw dataset: {df.shape[0]} samples, {df.shape[1]} columns.")
df.head()


Loaded raw dataset: 150 samples, 5 columns.


   SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0            5.1           3.5            1.4           0.2  Iris-setosa
1            4.9           3.0            1.4           0.2  Iris-setosa
2            4.7           3.2            1.3           0.2  Iris-setosa
3            4.6           3.1            1.5           0.2  Iris-setosa
4            5.0           3.6            1.4           0.2  Iris-setosa


## 2. Target Variable Encoding (`LabelEncoder`)
Machine learning algorithms require numerical labels. We transform string classes (`Iris-setosa`, `Iris-versicolor`, `Iris-virginica`) into categorical integers (`0, 1, 2`) using `LabelEncoder`.


In [2]:
# Initialize LabelEncoder
le = LabelEncoder()
df["Species_encoded"] = le.fit_transform(df["Species"])

# Display mapping
mapping_df = pd.DataFrame({
    "Original Species": le.classes_,
    "Encoded Label": le.transform(le.classes_)
})
print("Label Encoding Mapping:")
print(mapping_df.to_string(index=False))

# Verify label counts
print("
Encoded Class Distribution:")
print(df["Species_encoded"].value_counts().sort_index())


Label Encoding Mapping:
Original Species  Encoded Label
     Iris-setosa              0
 Iris-versicolor              1
  Iris-virginica              2

Encoded Class Distribution:
Species_encoded
0    50
1    50
2    50
Name: count, dtype: int64


## 3. Outlier Detection & Analysis (IQR Method)
Using the Interquartile Range ($IQR = Q_3 - Q_1$), a data point is identified as a statistical outlier if:
$$x < Q_1 - 1.5 \times IQR \quad \text{or} \quad x > Q_3 + 1.5 \times IQR$$


In [3]:
features = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
outlier_summary = []

for feature in features:
    q1 = df[feature].quantile(0.25)
    q3 = df[feature].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)]
    outlier_summary.append({
        "Feature": feature,
        "Q1": round(q1, 3),
        "Q3": round(q3, 3),
        "IQR": round(iqr, 3),
        "Lower Bound": round(lower_bound, 3),
        "Upper Bound": round(upper_bound, 3),
        "Outlier Count": len(outliers)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df


         Feature     Q1    Q3    IQR  Lower Bound  Upper Bound  Outlier Count
0  SepalLengthCm  5.100  6.40  1.300        3.150        8.350              0
1   SepalWidthCm  2.800  3.30  0.500        2.050        4.050              4
2  PetalLengthCm  1.600  5.10  3.500       -3.650       10.350              0
3   PetalWidthCm  0.300  1.80  1.500       -1.950        4.050              0


### Inspection of `SepalWidthCm` Statistical Outliers
Let us examine the exact records flagged in `SepalWidthCm`:


In [4]:
# Extract the 4 outlier rows
sw_q1 = df["SepalWidthCm"].quantile(0.25)
sw_q3 = df["SepalWidthCm"].quantile(0.75)
sw_iqr = sw_q3 - sw_q1
sw_outliers = df[(df["SepalWidthCm"] < sw_q1 - 1.5 * sw_iqr) | (df["SepalWidthCm"] > sw_q3 + 1.5 * sw_iqr)]
sw_outliers[["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm", "Species"]]


    SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm          Species
15            5.7           4.4            1.5           0.4      Iris-setosa
32            5.2           4.1            1.5           0.1      Iris-setosa
33            5.5           4.2            1.4           0.2      Iris-setosa
60            5.0           2.0            3.5           1.0  Iris-versicolor


### Outlier Assessment & Handling Decision
1. **Biological Reality vs. Sensor Noise**: The 4 outlier instances are valid biological measurements. `Iris-setosa` naturally possesses wide sepals (3 specimens measuring 4.1 cm, 4.2 cm, and 4.4 cm), while `Iris-versicolor` row 60 has a sepal width of 2.0 cm.
2. **Preservation Rationale**: In a compact dataset of 150 instances (50 per class), dropping genuine biological specimens would artificially truncate natural variance and risk over-optimistic generalization metrics.
3. **Treatment Decision**: We **retain** all 150 instances without alteration, relying on `StandardScaler` (Thursday) to normalize the distribution variance while preserving natural biological boundary morphology.


## 4. Train / Test Split (80% Train, 20% Test)
We split the features `X` and target `y` using an 80/20 ratio with fixed `random_state=42` to guarantee reproducible partitions.


In [5]:
# Define Feature Matrix X and Target Series y
X = df[features]
y = pd.Series(df["Species_encoded"], name="Species")

# Perform 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")
print("
Training Set Class Breakdown:")
print(y_train.value_counts().sort_index())
print("
Test Set Class Breakdown:")
print(y_test.value_counts().sort_index())


X_train shape: (120, 4), y_train shape: (120,)
X_test shape:  (30, 4), y_test shape:  (30,)

Training Set Class Breakdown:
Species
0    40
1    41
2    39
Name: count, dtype: int64

Test Set Class Breakdown:
Species
0    10
1     9
2    11
Name: count, dtype: int64


## 5. Exporting Preprocessed Datasets
We export the train and test partitions to the `outputs/` directory as requested.


In [6]:
# Export datasets without index
X_train.to_csv(outputs_dir / "X_train.csv", index=False)
X_test.to_csv(outputs_dir / "X_test.csv", index=False)
y_train.to_csv(outputs_dir / "y_train.csv", index=False)
y_test.to_csv(outputs_dir / "y_test.csv", index=False)

print("Preprocessed files successfully written to outputs/:")
for fname in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]:
    p = outputs_dir / fname
    print(f"  - {p.name} ({p.stat().st_size} bytes)")


Preprocessed files successfully written to outputs/:
  - X_train.csv (2095 bytes)
  - X_test.csv (565 bytes)
  - y_train.csv (369 bytes)
  - y_test.csv (99 bytes)


## 6. Wednesday Preprocessing Summary & Next Steps

| Deliverable Item | Status | Details |
| :--- | :--- | :--- |
| **Species Label Encoding** | Complete | Mapped to `0: Setosa`, `1: Versicolor`, `2: Virginica` |
| **Outlier Detection** | Complete | 4 points in `SepalWidthCm` evaluated and preserved as natural biology |
| **Train/Test Partitioning** | Complete | 120 train (80%), 30 test (20%), `random_state=42` |
| **Exported Files** | Complete | `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` |

### Hand-off to Thursday (Feature Scaling & Baseline Model):
The preprocessed partitions are ready for `StandardScaler` feature standardization and baseline `LogisticRegression` classification in `scripts/train_baseline_classifier.py`.
